# District Concentration, Dominance, and Rank Mobility in Migration Origins

Main content:
- Reshape district rank data into a long panel
- Compute concentration metrics (Top5 share, HHI, entropy, dominance ratios)
- Build a rank mobility transition matrix
- Measure district persistence and classify structural hubs vs transient spikes

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import theilslopes

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid', context='talk')

In [ ]:
# Configuration
DATA_PATH = Path('SriLanka_Migration_Dinura_Chanupa.csv')
EXPECTED_START_YEAR = 1994
EXPECTED_END_YEAR = 2025

REQUIRED_COLUMNS = [
    'year',
    'top1', 'top1_perc',
    'top2', 'top2_perc',
    'top3', 'top3_perc',
    'top4', 'top4_perc',
    'top5', 'top5_perc',
]

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Data file not found: {DATA_PATH}')

df_wide = pd.read_csv(DATA_PATH)
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_wide.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df_wide = df_wide[REQUIRED_COLUMNS].copy().sort_values('year').reset_index(drop=True)
df_wide.head()

In [ ]:
# Basic validation checks
expected_years = set(range(EXPECTED_START_YEAR, EXPECTED_END_YEAR + 1))
observed_years = set(df_wide['year'].astype(int).tolist())

validation = pd.DataFrame([
    ['Year continuity 1994-2025', expected_years == observed_years],
    ['Duplicate years', df_wide['year'].duplicated().sum() == 0],
    ['Share bounds [0, 100]', all((((df_wide[f'top{i}_perc'] >= 0) & (df_wide[f'top{i}_perc'] <= 100)).all() for i in range(1, 6)))],
    ['District labels non-null', not df_wide[[f'top{i}' for i in range(1, 6)]].isna().any().any()]
], columns=['check', 'passed'])

display(validation)
if not validation['passed'].all():
    raise ValueError('Validation failed. Please inspect the table above.')

## 1. Reshape to Long Panel

In [ ]:
parts = []
for rank in range(1, 6):
    temp = df_wide[['year', f'top{rank}', f'top{rank}_perc']].copy()
    temp['rank'] = rank
    temp.columns = ['year', 'district', 'emigration_share', 'rank']
    parts.append(temp)

df_long = pd.concat(parts, ignore_index=True).sort_values(['year', 'rank']).reset_index(drop=True)
display(df_long.head(12))

rows_per_year = df_long.groupby('year').size()
assert (rows_per_year == 5).all(), 'Each year must contain exactly five ranked rows.'

## 2. Concentration and Dominance Metrics

In [ ]:
metrics = df_wide[['year']].copy()
metrics['top5_share'] = sum(df_wide[f'top{i}_perc'] for i in range(1, 6))
metrics['dominance_1_5'] = df_wide['top1_perc'] / metrics['top5_share']
metrics['dominance_2_5'] = (df_wide['top1_perc'] + df_wide['top2_perc']) / metrics['top5_share']

hhi_vals = []
entropy_vals = []

for year in metrics['year']:
    shares = df_long.loc[df_long['year'] == year, 'emigration_share'].to_numpy(dtype=float)
    p = shares / shares.sum()

    hhi = float(np.sum(p ** 2) * 10000)
    hhi_vals.append(hhi)

    p_nz = p[p > 0]
    k = len(p_nz)
    entropy = float(-np.sum(p_nz * np.log(p_nz)) / np.log(k)) if k > 1 else 0.0
    entropy_vals.append(entropy)

metrics['hhi_top5_norm'] = hhi_vals
metrics['entropy_top5_norm'] = entropy_vals

display(metrics.head())

## 3. Rank Mobility Matrix

In [ ]:
years = sorted(df_long['year'].unique())
counts = np.zeros((5, 5), dtype=float)

for i in range(len(years) - 1):
    y0, y1 = years[i], years[i + 1]
    r0 = df_long[df_long['year'] == y0][['district', 'rank']].set_index('district')['rank']
    r1 = df_long[df_long['year'] == y1][['district', 'rank']].set_index('district')['rank']

    for district in r0.index.intersection(r1.index):
        from_rank = int(r0[district]) - 1
        to_rank = int(r1[district]) - 1
        counts[from_rank, to_rank] += 1

row_sums = counts.sum(axis=1, keepdims=True)
transition = np.divide(counts, row_sums, out=np.zeros_like(counts), where=row_sums != 0)

rank_names = [f'Rank_{i}' for i in range(1, 6)]
transition_df = pd.DataFrame(transition, index=rank_names, columns=rank_names)
display(transition_df.round(3))

## 4. Persistence and District Classification

In [ ]:
all_years = list(range(EXPECTED_START_YEAR, EXPECTED_END_YEAR + 1))
n_years = len(all_years)

def max_consecutive_true(flags):
    best, current = 0, 0
    for flag in flags:
        if flag:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best

rows = []
for district in sorted(df_long['district'].unique()):
    sub = df_long[df_long['district'] == district].copy()
    rank_by_year = {int(r.year): int(r.rank) for r in sub.itertuples(index=False)}
    share_by_year = {int(r.year): float(r.emigration_share) for r in sub.itertuples(index=False)}

    in_top5 = [year in rank_by_year for year in all_years]
    ranks_present = [rank_by_year[year] for year in all_years if year in rank_by_year]
    shares_present = [share_by_year[year] for year in all_years if year in share_by_year]

    rows.append({
        'district': district,
        'tenure_years_top5': int(sum(in_top5)),
        'tenure_pct_top5': float(sum(in_top5)) / n_years,
        'avg_rank_when_present': float(np.mean(ranks_present)) if ranks_present else np.nan,
        'avg_share_when_present': float(np.mean(shares_present)) if shares_present else np.nan,
        'std_share_when_present': float(np.std(shares_present, ddof=0)) if shares_present else np.nan,
        'max_consecutive_top5': max_consecutive_true(in_top5)
    })

persistence = pd.DataFrame(rows).sort_values(['tenure_pct_top5', 'avg_share_when_present'], ascending=[False, False])

# Practical rule-based classification
def classify(row):
    hub = (
        row['tenure_pct_top5'] >= 0.75 and
        row['avg_rank_when_present'] <= 3.0 and
        row['avg_share_when_present'] >= 7.5 and
        row['std_share_when_present'] <= 2.5
    )
    if hub:
        return 'Structural Hub'

    transient = (
        row['tenure_pct_top5'] <= 0.40 and
        row['avg_rank_when_present'] >= 4.5 and
        row['avg_share_when_present'] <= 6.5 and
        row['max_consecutive_top5'] <= 4
    )
    if transient:
        return 'Transient Spike'

    return 'Intermediate'

persistence['classification'] = persistence.apply(classify, axis=1)
display(persistence)

## 5. Trend Estimation

In [ ]:
def fit_ols_slope(y, x):
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    slope = float(model.params.iloc[1])
    ci_low = float(model.conf_int().iloc[1, 0])
    ci_high = float(model.conf_int().iloc[1, 1])
    return slope, ci_low, ci_high

tmp = metrics.copy()
tmp['t'] = np.arange(len(tmp), dtype=float)
tmp['t_centered'] = tmp['t'] - tmp['t'].mean()

trend_rows = []
for variable in ['top5_share', 'hhi_top5_norm', 'entropy_top5_norm', 'dominance_1_5', 'dominance_2_5']:
    ols_slope, ci_l, ci_h = fit_ols_slope(tmp[variable], tmp['t_centered'])
    ts_slope, _, ts_l, ts_h = theilslopes(tmp[variable].values, tmp['t'].values, alpha=0.95)
    trend_rows.append({
        'metric': variable,
        'ols_slope': ols_slope,
        'ols_ci_low': ci_l,
        'ols_ci_high': ci_h,
        'theil_sen_slope': float(ts_slope),
        'theil_sen_ci_low': float(ts_l),
        'theil_sen_ci_high': float(ts_h)
    })

trend_table = pd.DataFrame(trend_rows)
display(trend_table.round(5))

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(metrics['year'], metrics['top5_share'], marker='o')
axes[0, 0].set_title('Top5 Share (%) over Time')
axes[0, 0].set_xlabel('Year')

axes[0, 1].plot(metrics['year'], metrics['hhi_top5_norm'], marker='o', color='tab:orange')
axes[0, 1].set_title('HHI (Top5-normalized) over Time')
axes[0, 1].set_xlabel('Year')

axes[1, 0].plot(metrics['year'], metrics['dominance_1_5'], marker='o', label='Top1/Top5')
axes[1, 0].plot(metrics['year'], metrics['dominance_2_5'], marker='s', label='Top2/Top5')
axes[1, 0].set_title('Dominance Ratios over Time')
axes[1, 0].set_xlabel('Year')
axes[1, 0].legend()

sns.heatmap(transition_df, annot=True, fmt='.2f', cmap='Blues', ax=axes[1, 1], cbar_kws={'label': 'Transition probability'})
axes[1, 1].set_title('Rank Transition Probability Matrix')
axes[1, 1].set_xlabel('To rank')
axes[1, 1].set_ylabel('From rank')

plt.tight_layout()
plt.show()

In [ ]:
# District persistence heatmap (0 means district not in Top5 for that year)
rank_panel = df_long.pivot_table(index='district', columns='year', values='rank', aggfunc='first').fillna(0)
rank_panel = rank_panel.reindex(persistence.sort_values('tenure_pct_top5', ascending=False)['district'])

plt.figure(figsize=(16, 8))
sns.heatmap(rank_panel, cmap='YlGnBu', cbar_kws={'label': 'Rank (0 = out of Top5)'})
plt.title('District Rank Persistence Heatmap')
plt.xlabel('Year')
plt.ylabel('District')
plt.tight_layout()
plt.show()

## 7. Key Findings

In [ ]:
latest = metrics.sort_values('year').iloc[-1]
hub_count = int((persistence['classification'] == 'Structural Hub').sum())
transient_count = int((persistence['classification'] == 'Transient Spike').sum())

hhi_slope = float(trend_table.loc[trend_table['metric'] == 'hhi_top5_norm', 'theil_sen_slope'].iloc[0])
dom_slope = float(trend_table.loc[trend_table['metric'] == 'dominance_1_5', 'theil_sen_slope'].iloc[0])

print('District Concentration and Rank Mobility: Key Findings')
print(f'- Latest Top5 share ({int(latest["year"])}): {latest["top5_share"]:.2f}%')
print(f'- Latest HHI ({int(latest["year"])}): {latest["hhi_top5_norm"]:.2f}')
print(f'- Theil-Sen slope (HHI): {hhi_slope:.3f} per year')
print(f'- Theil-Sen slope (Top1/Top5 dominance): {dom_slope:.5f} per year')
print(f'- Structural hubs identified: {hub_count}')
print(f'- Transient spikes identified: {transient_count}')
print('- Interpretation:', 'concentration is strengthening over time.' if hhi_slope > 0 else 'concentration is easing over time.')